# 06 — Comparación de Modelos y Ablation Study

Comparamos tres backbones para la tarea de autenticación de documentos:

| Modelo | Params | Ventaja |
|--------|--------|---------|
| EfficientNet-B0 | ~5.3M | Balance precisión/eficiencia (modelo de producción) |
| ResNet-18 | ~11.2M | Baseline clásico, bien estudiado |
| MobileNetV3-Small | ~2.5M | Edge/mobile, latencia mínima |

**Metodología**: todos los modelos reciben el mismo preprocesamiento, el mismo split test y un único umbral de decisión (0.5). Se registran métricas de clasificación y latencia en MLflow.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

from src.models.architectures import build_classifier, Backbone
from src.models.comparator import ModelComparator

print(f'torch {torch.__version__}')
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {DEVICE}')

## 1. Datos sintéticos para demostración

En un pipeline real se usaría `create_dataloaders(data_dir)`.  
Aquí generamos tensores aleatorios para demostrar el framework de comparación sin requerir datos reales ni checkpoints entrenados.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

N_TEST = 200
torch.manual_seed(42)

images = torch.randn(N_TEST, 3, 224, 224)
labels = torch.randint(0, 2, (N_TEST,)).float()

test_ds = TensorDataset(images, labels)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f'Test set: {N_TEST} samples ({int(labels.sum())} forged, {N_TEST - int(labels.sum())} authentic)')

## 2. Instanciar modelos

> **Nota**: modelos con pesos pre-entrenados en ImageNet (`pretrained=True`).  
> Para una comparación real, cargar checkpoints entrenados con `DocumentClassifierV2.load(path)`.

In [ ]:
backbones: list[Backbone] = ['efficientnet_b0', 'resnet18', 'mobilenet_v3_small']

models_dict = {}
for name in backbones:
    m = build_classifier(backbone=name, pretrained=True)
    m.eval()
    models_dict[name] = m
    print(f'{name:25s}  {m.total_params():>10,} params')

## 3. Comparación con ModelComparator

In [ ]:
comparator = ModelComparator(
    test_loader=test_loader,
    device='cpu',   # latencia siempre en CPU para comparación justa
    mlflow_tracking_uri=str(ROOT / 'mlflow.db'),
    n_warmup=3,
    n_latency_reps=20,
)

for name, model in models_dict.items():
    comparator.add_model(name, model)

results = comparator.run(mlflow_run_name='ablation_backbone_v1')
df = pd.DataFrame(results).set_index('name')
df

## 4. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Comparación de backbones — datos sintéticos\n(con datos reales los valores de clasificación serían significativos)', 
             fontsize=10, color='gray')

colors_map = {
    'efficientnet_b0': '#2ecc71',
    'resnet18': '#3498db',
    'mobilenet_v3_small': '#e67e22',
}
bar_colors = [colors_map[n] for n in df.index]

# --- AUC-ROC ---
ax = axes[0]
ax.bar(df.index, df['auc_roc'], color=bar_colors, width=0.5)
ax.set_title('AUC-ROC')
ax.set_ylim(0, 1)
ax.axhline(0.5, ls='--', color='gray', lw=0.8, label='random')
ax.set_xticklabels(df.index, rotation=20, ha='right')
ax.legend(fontsize=8)

# --- Latencia ---
ax = axes[1]
ax.bar(df.index, df['avg_inference_ms'], color=bar_colors, width=0.5)
ax.set_title('Latencia media (ms) — CPU, batch=1')
ax.set_xticklabels(df.index, rotation=20, ha='right')

# --- Parámetros ---
ax = axes[2]
params_m = df['total_params'] / 1e6
ax.bar(df.index, params_m, color=bar_colors, width=0.5)
ax.set_title('Parámetros totales (M)')
ax.set_xticklabels(df.index, rotation=20, ha='right')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f M'))

plt.tight_layout()
out_dir = ROOT / 'reports' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / '06_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Tabla resumen

In [ ]:
summary = df[['accuracy', 'f1', 'auc_roc', 'precision', 'recall', 'avg_inference_ms', 'total_params']].copy()
summary['total_params'] = (summary['total_params'] / 1e6).map('{:.2f}M'.format)
summary['avg_inference_ms'] = summary['avg_inference_ms'].map('{:.1f}'.format)
summary.columns = ['Accuracy', 'F1', 'AUC-ROC', 'Precision', 'Recall', 'Latencia (ms)', 'Params']

# Highlight max in each metric column (excluding Latencia and Params)
styled = summary.style.highlight_max(
    subset=['Accuracy', 'F1', 'AUC-ROC', 'Precision', 'Recall'],
    color='#d4efdf',
).format({
    'Accuracy': '{:.3f}',
    'F1': '{:.3f}',
    'AUC-ROC': '{:.3f}',
    'Precision': '{:.3f}',
    'Recall': '{:.3f}',
})
styled

## 6. Conclusiones del ablation study

> Los valores de clasificación sobre datos sintéticos son ≈0.5 (aleatorio) por diseño.  
> Las conclusiones relevantes son sobre **latencia** y **tamaño de modelo**:

| Criterio | Recomendación |
|----------|---------------|
| **Máxima precisión** | EfficientNet-B0 — mejor trade-off accuracy/params en tareas de visión de detalle fino |
| **Edge / tiempo real** | MobileNetV3-Small — menor latencia, 2× menos parámetros que EfficientNet-B0 |
| **Baseline / interpretabilidad** | ResNet-18 — arquitectura bien documentada, útil como control experimental |

**Modelo de producción elegido**: `efficientnet_b0` — históricamente superior en benchmarks de clasificación fina  
con objetos pequeños y texturas (sellos fiscales), y con latencia aceptable para uso interactivo (<50ms en CPU).

Los resultados completos están registrados en MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow.db`).